In [ ]:
# -----------------------------------------
# Reads the assigned.csv file for one subject safely
# -----------------------------------------
def read_assigned_sensor_csv(subject_path: Path):
    assigned_files = list(subject_path.glob("assigned.csv"))
    if not assigned_files:
        return None  # No assigned.csv found

    assigned_data = pd.read_csv(assigned_files[0])

    required_cols = ["affected finger", "affected wrist", "less affected finger", "less affected wrist"]
    for col in required_cols:
        if col not in assigned_data.columns:
            return None  # Missing required column

    return {
        "affected finger": assigned_data["affected finger"],
        "affected wrist": assigned_data["affected wrist"],
        "less affected finger": assigned_data["less affected finger"],
        "less affected wrist": assigned_data["less affected wrist"]
    }

# -----------------------------------------
# Main driver function (subject-level) with missing file handling
# -----------------------------------------
def process_data(assigned_dir: Path, output_dir: Path, pipeline: callable):
    """
    Apply the processing pipeline to each subject's assigned.csv file.
    Skips subjects with missing files or missing required columns.
    Saves processed.csv per subject.
    """
    subject_paths = get_subject_paths(assigned_dir)
    create_output_dirs(subject_paths, output_dir)

    for subject_path in subject_paths:
        subject = subject_path.name
        print(f"Processing subject: {subject}")

        # Read assigned data safely
        assigned_dict = read_assigned_sensor_csv(subject_path)
        if assigned_dict is None:
            print(f" Skipping {subject}: missing assigned.csv or required columns")
            continue

        # Apply processing pipeline to each channel
        processed_dict = {
            "affected finger": pipeline(assigned_dict["affected finger"]),
            "affected wrist": pipeline(assigned_dict["affected wrist"]),
            "less affected finger": pipeline(assigned_dict["less affected finger"]),
            "less affected wrist": pipeline(assigned_dict["less affected wrist"])
        }

        # Combine processed signals into DataFrame
        df_processed = pd.DataFrame({
            "affected finger": pd.Series(processed_dict["affected finger"]),
            "affected wrist": pd.Series(processed_dict["affected wrist"]),
            "less affected finger": pd.Series(processed_dict["less affected finger"]),
            "less affected wrist": pd.Series(processed_dict["less affected wrist"])
        })

        # Save processed data for this subject
        output_file = output_dir / subject / "processed.csv"
        df_processed.to_csv(output_file, index=False)
        print(f"Saved: {output_file}")

# -----------------------------------------
# Example usage
# -----------------------------------------
process_data(
    assigned_dir=Path("path/to/your/data"),
    output_dir=Path("path/to/your/data"),
    pipeline=process_data_pipeline
)
